# Day 2 — Lecture 3: For Loops


## Goals for this lecture

> Teal **Tasks** are required. Two Tasks are **pair+share** (§1.2, §1.4) — allow ~2 min partner discussion before class debrief. Red content is in `D2_Dive_Deeper_3.ipynb`.

This lecture teaches **`for` loops only**. Python also has `while` loops; those are optional reading in `D2_Dive_Deeper_3.ipynb`.

### After the core lecture, you should be able to:

1. Write a `for` loop over indices with `range(len(arr))`
2. Combine a `for` loop with an `if` filter on real lidar timestamps
3. Load lidar wind arrays from CSV and interpret `shape`, slices, and `nanmin`/`nanmax`
4. Explain what a June 2020 hourly-mean wind array represents
5. Format a matplotlib line plot with labels, title, and grid

### Optional extensions

| Notebook | Contents |
|----------|----------|
| `D2_Dive_Deeper_3.ipynb` | `for` loop + `if` drill, `while` loops, raw June plot, refactor `for` loops, trend line |
| `D2_Challenge_3.ipynb` | Seasonal daily-averaged winds with `for` loops (afternoon) |

Compare optional work with instructor solutions when finished.


## Notebook and Learning Structure

Any text in black/white will be instruction and guidance and will usually start with a section number.<br>
<font color="#0F766E"> Any text in teal will be tasks to do and start with "Task".</font><br>
<font color="#B91C1C"> Any text in red will be optional challenges and advanced concepts for anyone looking to try more.</font>


**Import packages:**

In [ ]:
# Mount the github repo to access data
from google.colab import drive
drive.mount('/content/drive')

# Path to shared bootcamp data (repo-root Datasets/)
DATA_FOLDER = "/content/drive/MyDrive/python_bootcamp/python_bootcamp_for_earth_science/Datasets/"
FIGURES_FOLDER = "/content/drive/MyDrive/python_bootcamp/python_bootcamp_for_earth_science/Figures/"


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## 1.0 Bridge: from `if` to `for` loops at scale

This morning in **Lecture 2** you classified wind speeds and seasons with `if`, `elif`, and `in`, and combined conditions with `and` (for example, `year == 2020 and month == 6`).

Now we use **`for` loops** over **thousands** of lidar records and put `if` **inside** the loop to keep only the timestamps we care about. We assume you are comfortable with basic `if` logic — we will not re-teach it here.


### 1.1 Basic `for` loops


In [ ]:
# Make a numpy array of fruit names (an array of strings)
a = np.array(['apple', 'orange', 'pear', 'watermelon', 'grapes'])

# Loop through the array to see what it contains
for i in a:
    print('The value in this spot is ' + str(i))


When we work with data, we often want to loop through each **index** of an array as opposed to each **value**.

Instead of:
> `for i in a:` — where `i` takes values `apple`, `orange`, `pear`, ...

We write:
> `for i in range(len(a)):` — where `i` takes values `0, 1, 2, ...`

**Note:** `range()` starts counting from 0 and stops before `len(a)`.


<font color="#0F766E">**Task (pair+share):** Look at the code in the next cell, but **do not run it yet**. What will print? Write your prediction in the cell after that, then **compare with a neighbor** before we run it together.</font>


In [ ]:
# Make a numpy array of values
a = np.array(['apple', 'orange', 'pear', 'watermelon', 'grapes'])

for i in range(len(a)):
    print('The value at index ' + str(i) + ' is ' + str(a[i]))


*(Write your prediction here — double-click to edit)*


### 1.2 `for` loop practice with `range(len)`


In [ ]:
# Print which fruit is stored at index 1
a[1]


In [ ]:
# Exclude the last element when printing array values
a = np.array(['apple', 'orange', 'pear', 'watermelon', 'grapes'])

for i in range(len(a) - 1):
    print('The value at index ' + str(i) + ' is ' + str(a[i]))


Notice how *grapes*, the last value in the array, was not printed.


In [ ]:
# Create an array
a = np.array([4, 5, 20, 7, 25, 15, 9, 10])


<font color="#0F766E">**Task:** Using the array above, write a `for` loop that prints each value multiplied by 2.</font>


In [ ]:
# Your code here


### 1.3 Real lidar data


In [ ]:
from IPython.display import Image, display
display(Image(FIGURES_FOLDER + "lidar_array_structure.png"))


The diagram above shows the basics of how Lidar works to measure different things in Earth's atmosphere. You do not need to understand how this instrument works but we will be using data taken from this instrument in the code below.

In [ ]:
# Uploading data — don't need to worry about how to write any of this code yet!
lidar_winds = pd.read_csv(DATA_FOLDER + "lidar_winds_short.csv")
lidar_winds['timestamp'] = pd.to_datetime(lidar_winds['timestamp'])

# Separate our year, month, day, time
lidar_winds['Year'] = pd.DatetimeIndex(lidar_winds['timestamp']).year
lidar_winds['Month'] = pd.DatetimeIndex(lidar_winds['timestamp']).month
lidar_winds['Day'] = pd.DatetimeIndex(lidar_winds['timestamp']).day
lidar_winds['Time'] = pd.DatetimeIndex(lidar_winds['timestamp']).time

# Create numpy arrays of values from dataarray
wind18m = np.asarray(lidar_winds['wspd18m'])
year = np.asarray(lidar_winds['Year'])
month = np.asarray(lidar_winds['Month'])
day = np.asarray(lidar_winds['Day'])
time = np.asarray(lidar_winds['Time'])


For every record in the lidar data we are working with, we have a timestamp and an 18 m wind speed (`wind18m`). We also extracted `year`, `month`, `day`, and `time` arrays. Let's investigate what this data looks like!


In [ ]:
# Print the shape of the wind18m data
wind18m.shape


In [ ]:
# Print the first 4 values in wind18m
print(wind18m[0:4])

# Notice how 'wind18m[0:4]' returns the values at indices 0, 1, 2, and 3
# but NOT the value at index 4


In [ ]:
# Print the first and last timestamp components
print(year[0], month[0], day[0], time[0])
print(year[-1], month[-1], day[-1], time[-1])


In [ ]:
# np.nanmin() and np.nanmax() ignore missing (NaN) entries
print(np.nanmin(wind18m))
print(np.nanmax(wind18m))


<font color="#0F766E">**Task:** After running the cells above, summarize what you learned about the lidar arrays in one short paragraph: array size (`shape`), a sample of wind values, the time span of the data, and the min/max wind speeds.</font>


*(Write your answer here — double-click to edit)*


### 1.4 Filter June 2020 and compute hourly means


We will do this in two phases:

1. **Filter** — use a `for` loop over every record and keep only June 2020 rows using `if year_i == 2020 and month_i == 6` (the same compound condition you practiced in Lecture 2 §1.6).
2. **Aggregate** — use a second `for` loop over 6 ten-minute samples at a time and compute an hourly mean with `np.nanmean`.

Walk through the commented code below with your instructor (~5 min), then run it.


<font color="#0F766E">**Task (pair+share):** Read the code in the next cell, but **do not run it yet**. What do you expect `wind_06_2020` and `hourly_wind_06_2020` to contain? Write your prediction below, then **compare with a neighbor** before we run it together.</font>


In [ ]:
# Let the variable num_times represent the number of times we have in our data
num_times = len(wind18m)

# Initialize empty numpy arrays to store the date, time, and wind speed for days in June 2020
day_06_2020 = np.array([])
time_06_2020 = np.array([])
wind_06_2020 = np.array([])

# For a range of consecutive values from zero to the number of values in the wind array, do the following
for i in range(num_times):

    # Save the information we need for the next steps of the loop into variables
    year_i = year[i]
    month_i = month[i]
    day_i = day[i]
    time_i = time[i]
    wdsp = wind18m[i]

    # If the year is 2020 and the month is June, then do the following
    if year_i == 2020 and month_i == 6:
        # Add these values to our arrays for June
        day_06_2020 = np.append(day_06_2020, day_i)
        time_06_2020 = np.append(time_06_2020, time_i)
        wind_06_2020 = np.append(wind_06_2020, wdsp)


# Initialize new arrays to store the hourly-averaged wind speed and the day/hour corresponding
# to this average. The day/hour will act as our x while hourly-averaged wind speed acts as our
# y for a y vs x plot.
day_hour_06_2020 = np.array([])
hourly_wind_06_2020 = np.array([])

# Let this variable represent the number of times we have in our data from June 2020
num_times_06_2020 = len(wind_06_2020)

# For a range of consecutive values from zero to the total number of hourly timesteps in June 2020
# (the number of 10 minutely timesteps divided by 6, since there are six 10-minute intervals in an hour)
for j in range(int(num_times_06_2020 / 6)):

    # Make a separate counter k, which counts through all of the 10-minutely June 2020 timesteps in each hour
    k = j * 6

    # Make a new string day_hour that stores both the day and hour of the hour-averaged wind speed
    day_hour = str(day_06_2020[k]) + ' ' + str(time_06_2020[k])

    # Calculate the wind speed for this hour as the average wind speed across the k, k+1, k+2,...,k+5
    # timesteps of the 10 minute data
    hourly_wdsp = np.nanmean(wind_06_2020[k:k + 6])

    # Save the day/hour string constructed above and the average of wind speed found for the current hour
    # into our new arrays
    day_hour_06_2020 = np.append(day_hour_06_2020, day_hour)
    hourly_wind_06_2020 = np.append(hourly_wind_06_2020, hourly_wdsp)


*(Write your prediction here — double-click to edit)*


<font color="#0F766E">**Task:** Run the extraction cell above. In your own words, what does the resulting array `hourly_wind_06_2020` represent?</font>


*(Write your answer here — double-click to edit)*


### 1.5 Plot hourly June 2020 winds


We can visualize our hourly wind data using Matplotlib:


In [ ]:
# Create the figure
fig = plt.figure(figsize=(12, 6))

# Add one subplot to figure
ax = fig.add_subplot(1, 1, 1)

# Plot time on the x axis, wind on the y
plt.plot(day_hour_06_2020, hourly_wind_06_2020, color='black')

# Only label every 24 ticks (daily), otherwise it gets too crowded
ax.set_xticks(ax.get_xticks()[::24])

# Rotate tick labels for better spacing/viewing
ax.tick_params(axis='x', rotation=45)

plt.show()


<font color="#0F766E">**Task:** Edit the code in the next cell so the output matches the reference image below it. Hint: search for matplotlib `set_title`, `set_xlabel`, `set_ylabel`, and `grid`.</font>


In [ ]:
# Edit this cell so the plot matches the reference image below.
# Hint: search for matplotlib set_title, set_xlabel, set_ylabel, and grid.

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(1, 1, 1)

plt.plot(day_hour_06_2020, hourly_wind_06_2020, color='black')
ax.set_xticks(ax.get_xticks()[::24])
ax.tick_params(axis='x', rotation=45)

# Your formatting edits here


plt.show()


In [ ]:
from IPython.display import Image, display
display(Image(FIGURES_FOLDER + "wind_plot_reference.png"))


### 1.6 Wrap-up

**Core lecture ends here (~65 min).**

You used `if` with `and` this morning; today you applied that filter inside a `for` loop over every lidar record.

**Afternoon / extra time:**
1. `D2_Challenge_3.ipynb` — seasonal daily-averaged winds (reuse season month lists from Lecture 2)
2. `D2_Dive_Deeper_3.ipynb` — `for` loop + `if` drill, `while` loops, raw June plot, refactor `for` loops, trend line
3. Compare your work with instructor solutions when finished
